In [1]:
import pandas as pd
import mibian
import numpy as np

# Load the master file we just created
df = pd.read_csv("../data/nifty_merged_5min.csv", index_col=0, parse_dates=True)

# Task 2.2: Greeks Calculation Logic
def calculate_greeks(row):
    # mibian.BS([Underlying, Strike, InterestRate, DaysToExpiry], volatility=IV)
    # We use 6.5 as the Interest Rate (Risk-Free Rate) as per Task 2.2
    
    # Call Greeks
    c = mibian.BS([row['Close'], row['ATM_Strike'], 6.5, 5], volatility=row['IV_CE']*100)
    # Put Greeks
    p = mibian.BS([row['Close'], row['ATM_Strike'], 6.5, 5], volatility=row['IV_PE']*100)
    
    return pd.Series({
        'Delta_CE': c.callDelta,
        'Delta_PE': p.putDelta,
        'Gamma': c.gamma,
        'Theta_CE': c.callTheta,
        'Theta_PE': p.putTheta,
        'Vega': c.vega
    })

print("Calculating Greeks... this might take a minute.")
greeks = df.apply(calculate_greeks, axis=1)
df = pd.concat([df, greeks], axis=1)

# Task 2.3: Derived Features
df['IV_Spread'] = df['IV_CE'] - df['IV_PE'] # IV difference [cite: 54]
df['PCR_OI'] = df['OI_PE'] / df['OI_CE'] # Sentiment indicator 
df['Futures_Basis'] = (df['Close_fut'] - df['Close']) / df['Close'] # Basis [cite: 57]
df['Gamma_Exposure'] = df['Close'] * df['Gamma'] * (df['OI_CE'] + df['OI_PE']) # Risk [cite: 60]

# Task 2.4: Save the Final Feature Set
df.to_csv("../data/nifty_features_5min.csv")
print("Task 2 complete! Your features are ready for Regime Detection.")

Calculating Greeks... this might take a minute.


Task 2 complete! Your features are ready for Regime Detection.
